# XTTS-v2 — Клонирование голоса на русском языке

**Что делает этот ноутбук:**
- Клонирует голос по 6+ секундам аудио
- Сохраняет картавость и индивидуальные особенности речи
- Полная поддержка русского языка
- Работает бесплатно на Google Colab GPU

**Порядок:**
1. Шаг 1 — проверь GPU
2. Шаг 2 — установка (3-5 минут, один раз)
3. Шаг 3 — загрузка модели (скачает ~2GB)
4. Шаг 4 — загрузи свой голос (WAV или MP3, 6-30 секунд)
5. Шаг 5 — введи текст и получи результат
6. Шаг 6 — скачай файл

## Шаг 1: Проверь GPU
Должно написать "GPU найден!" и название карты. Если нет — **Среда выполнения → Сменить → T4 GPU**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    print(f'GPU найден: {result.stdout.strip()}')
else:
    print('GPU НЕ найден! Перейди: Среда выполнения → Сменить среду → T4 GPU')

## Шаг 2: Установка XTTS-v2
Одна команда — занимает 3-5 минут. Предупреждения (жёлтый/оранжевый текст) — нормально, ждём зелёную галочку.

In [ ]:
!pip install -q coqui-tts
!pip install -q "transformers>=4.46"
import os
os.environ["COQUI_TOS_AGREED"] = "1"
print("Установка завершена!")
print()
print("=" * 50)
print("ВАЖНО: после этого — Restart session!")
print("Runtime → Restart session")
print("После перезапуска запускай с Шага 3.")
print("=" * 50)

## Шаг 3: Загрузка модели XTTS-v2
Скачается ~2GB — займёт 2-5 минут в зависимости от скорости интернета.

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

# Патч: добавляем функцию isin_mps_friendly, убранную из новых transformers
import torch
import transformers.pytorch_utils as _pt_utils
if not hasattr(_pt_utils, 'isin_mps_friendly'):
    def _isin_mps_friendly(elements, test_elements):
        return torch.isin(elements, test_elements)
    _pt_utils.isin_mps_friendly = _isin_mps_friendly
    print("Патч совместимости применён.")

from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем: {device}")

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

os.makedirs("/content/output", exist_ok=True)
print("Модель готова к работе!")

## Шаг 4: Загрузи свой голос

**Требования к записи:**
- Формат: WAV или MP3
- Длина: 6–30 секунд (оптимально 10–15 сек)
- Без фоновой музыки и эха
- Говори чётко, с твоими обычными особенностями (картавость и т.д.)

**Текст для записи (много букв Р):**
> "Рыжая лиса прыгнула через реку. Красные розы растут в ряду. Прораб проверил работу рабочих. Три пирога с творогом."

In [ ]:
from google.colab import files

print("Выбери свой аудиофайл (WAV или MP3)...")
uploaded = files.upload()

for filename in uploaded.keys():
    ref_path = f"/content/reference_voice_{filename}"
    with open(ref_path, "wb") as f:
        f.write(uploaded[filename])
    print(f"Загружен: {filename} → сохранён как {ref_path}")

# Запоминаем путь к последнему загруженному файлу
reference_audio = ref_path
print(f"\nЭталонный голос: {reference_audio}")

## Шаг 5: Введи текст и синтезируй речь
Измени текст в ячейке ниже на свой, затем запусти.

In [ ]:
import IPython.display as ipd

# ==============================
# ИЗМЕНИ ЭТОТ ТЕКСТ НА СВОЙ
# ==============================
text = """Привет! Это клон моего голоса.
Рыжая лиса прыгнула через реку.
Красные розы растут в ряду."""

output_path = "/content/output/cloned_voice.wav"

# Синтез с клонированием голоса
tts.tts_to_file(
    text=text,
    speaker_wav=reference_audio,
    language="ru",
    file_path=output_path,
)

print("Готово! Слушаем результат:")
ipd.display(ipd.Audio(output_path))

## Шаг 6: Скачай результат

In [ ]:
from google.colab import files
files.download(output_path)
print("Файл скачан!")

## Шаг 6: Скачай результат

## Дополнительно: Попробуй разные варианты
Можно менять скорость и запускать снова